# Bu notebookda projede kullanılacak makine öğrenim süreçleri modeller ve searchler bulunmaktadır


vektörize edilmiş market databaseimiz için search meselesi


In [27]:
import joblib
import polars as pl
from rapidfuzz import fuzz
from sklearn.metrics.pairwise import cosine_similarity

def turkce_karakter_temizle(metin):
    harfler = {
        "ç": "c", "ş": "s", "ğ": "g", "ı": "i", "ö": "o", "ü": "u",
        "Ç": "c", "Ş": "s", "Ğ": "g", "İ": "i", "Ö": "o", "Ü": "u"
    }
    for tr, eng in harfler.items():
        metin = metin.replace(tr, eng)
    return metin.lower()

loaded_vectorizer = joblib.load('tfidf_vectorizer.pkl')
loaded_matrix = joblib.load('tfidf_matrix.pkl')
loaded_df = pl.read_parquet('cleaned_dataframe.parquet')

def hibrit_market_aramasi(malzeme_adi, top_n=3):
    temiz_malzeme = turkce_karakter_temizle(malzeme_adi)
    
    birlesik_malzeme = temiz_malzeme.replace(" ", "")
    
    vec_ayri = loaded_vectorizer.transform([temiz_malzeme])
    vec_birlesik = loaded_vectorizer.transform([birlesik_malzeme])
    
    skor_ayri = cosine_similarity(vec_ayri, loaded_matrix).flatten()
    skor_birlesik = cosine_similarity(vec_birlesik, loaded_matrix).flatten()
    
    if skor_birlesik.max() > skor_ayri.max():
        aktif_skorlar = skor_birlesik
        aktif_kelime = birlesik_malzeme
    else:
        aktif_skorlar = skor_ayri
        aktif_kelime = temiz_malzeme
        
    aday_indeksler = aktif_skorlar.argsort()[-20:][::-1]
    aday_listesi = []
    
    for idx in aday_indeksler:
        tfidf_skor = aktif_skorlar[idx]
        urun = loaded_df["ITEMNAME"][int(idx)]
        kat1 = loaded_df["CATEGORY1"][int(idx)]
        kat2 = loaded_df["CATEGORY2"][int(idx)]
        
        urun_temiz = turkce_karakter_temizle(urun)
        
        fuzzy_skor = fuzz.token_set_ratio(aktif_kelime, urun_temiz) / 100.0
        
        final_skor = (tfidf_skor * 0.3) + (fuzzy_skor * 0.7)
        
        aday_listesi.append({
            "urun": urun,
            "kat1": kat1,
            "kat2": kat2,
            "skor": final_skor,
            "detay": f"(TF-IDF: {tfidf_skor:.2f} | Fuzzy: {fuzzy_skor:.2f})"
        })
        
    aday_listesi = sorted(aday_listesi, key=lambda x: x["skor"], reverse=True)
    
    print(f"\n--- Girdi: '{malzeme_adi}' | Algoritmanın Kararı: '{aktif_kelime}' ---")
    for i in range(min(top_n, len(aday_listesi))):
        sonuc = aday_listesi[i]
        print(f"[{sonuc['skor']:.2f}] [{sonuc['kat1']} > {sonuc['kat2']}] {sonuc['urun']} {sonuc['detay']}")



hibrit_market_aramasi("1 adet hazır yufka")
hibrit_market_aramasi("1 adet domates")
hibrit_market_aramasi("1 adet yeşil biber")
hibrit_market_aramasi("1 adet kırmızı kapya biber")
hibrit_market_aramasi("150 g rendelenmiş kaşar peyniri")
hibrit_market_aramasi("pastırma")
hibrit_market_aramasi("sıvı yağ")



--- Girdi: '1 adet hazır yufka' | Algoritmanın Kararı: '1 adet hazir yufka' ---
[0.55] [KAHVALTILIK > UNLU MAMULLER] ALCAN YUFKA 1 KG (TF-IDF: 0.42 | Fuzzy: 0.61)
[0.54] [KAHVALTILIK > UNLU MAMULLER] NERGIS  YUFKA 1 KG (TF-IDF: 0.45 | Fuzzy: 0.58)
[0.54] [KAHVALTILIK > UNLU MAMULLER] NERGIS YUFKA 1 KG (TF-IDF: 0.45 | Fuzzy: 0.58)

--- Girdi: '1 adet domates' | Algoritmanın Kararı: '1 adet domates' ---
[0.79] [SEBZE > MANAV] DOMATES (TF-IDF: 0.31 | Fuzzy: 1.00)
[0.67] [SEBZE > MANAV] G. DOMATES  (TF-IDF: 0.31 | Fuzzy: 0.82)
[0.64] [GIDA > HAZIR YEMEK-KONSERVE-SALCA] IPEK SALCA DOMATES 1 KG (TF-IDF: 0.30 | Fuzzy: 0.78)

--- Girdi: '1 adet yeşil biber' | Algoritmanın Kararı: '1 adet yesil biber' ---
[0.81] [SEBZE > MANAV] BIBER YESIL  (TF-IDF: 0.35 | Fuzzy: 1.00)
[0.68] [SEBZE > MANAV] BIBER YESIL ACI (TF-IDF: 0.29 | Fuzzy: 0.85)
[0.61] [SEBZE > MANAV] GOLBASI BIBER YESIL ACI (TF-IDF: 0.26 | Fuzzy: 0.76)

--- Girdi: '1 adet kırmızı kapya biber' | Algoritmanın Kararı: '1 adet kirmizi kapy

kısaca yukarda ne yaptığımı anlatayım bizim yemek tariflerinden çıkaracağımız malzemeleri market database'imizde bulmamız lazımdı bunun için search algoritma modellerine ihtiyacımız vardı ilk adım olarak tf-idf ile kelimelerimizin matematiğini çıkardık sonrasında kosinüs benzerliği ile basit bi algoritma kurdum ancak domates - domates salçası aratmasında aynı sonuçlar geldi çünkü ikisi de kosinüs olarak neredeyse aynı ama farklı şeyler bunun sonucunda oyuna rapidfuzz geldi ve cümle uzunluklarını kattı sonra salça aratması yaptım alakasız şeyler geldi çünkü türkçe karakterleri anlamıyordu algoritma basit bi türkçe karakter dönüşümü ekledim bu sorun da böyle çözüldü sonrasında kara biber ve karabiber araması yaptım apayrı sonuçlar çıktı sonrasında boşluk için birleşik arama da ekledim